The Jobicy dataset contains very long job descriptions. 
    
Some descriptions exceed 14,000 characters.

These descriptions contain commas and other special characters. 
    
Although the CSV is correctly formatted, MySQL Workbench's Import Wizard can encounter problems when processing such data.

So we have imported our Jobicy data using Jupyter Notebook instead of the MySQL Workbench Import Wizard.

Before running: create the tables by running `sql/JobAvenir_01_Create_Database_MySQL.sql` in MySQL Workbench.

Step 1: Install the MySQL Python connector

In [ ]:
%pip install mysql-connector-python

Step 2: Connect Jupyter Notebook to MySQL

In [1]:
import mysql.connector
from getpass import getpass

connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password=getpass("Enter your MySQL password: "),
    database="jobavenir_db"
)

print("Connected to MySQL successfully!")

Enter your MySQL password:Sheetal@24 ········


Connected to MySQL successfully!


Step 3: Import the Jobicy CSV into MySQL

In [ ]:
import pandas as pd
from pathlib import Path

# 1 = replace any rows already in the staging tables with the current CSVs
#     (use after re-running 02_Jobicy_Data_Collection.ipynb; core tables are untouched)
# 0 = import only into empty staging tables
REPLACE_STAGING = 1

# Path is relative to this notebook (notebooks/)
CLEANED_FOLDER = Path("..") / "data" / "cleaned"
file_path = CLEANED_FOLDER / "jobicy_jobs_clean.csv"

# Read Jobicy CSV
df = pd.read_csv(file_path, dtype=str)

# Display dataset information
print("Total Records:", len(df))
print("Columns:", df.columns.tolist())

display(df.head())

# Required columns (written by 02_Jobicy_Data_Collection.ipynb)
columns = [
    "job_id",
    "job_title",
    "company_name",
    "location",
    "description_clean",
    "publication_date",
    "job_url",
    "job_level",
    "job_industry",
    "salary_min",
    "salary_max",
    "salary_currency",
    "salary_period",
    "searched_role",
    "collected_at",
    "last_seen_at"
]

# Ensure CSV contains the expected columns
assert list(df.columns) == columns, "CSV columns do not match! Re-run 02_Jobicy_Data_Collection.ipynb."

cursor = connection.cursor()

try:
    # The staging table must already have the new columns
    cursor.execute("SHOW COLUMNS FROM stg_jobicy_postings")
    table_columns = [row[0] for row in cursor.fetchall()]

    missing_columns = []
    for column in columns:
        if column not in table_columns:
            missing_columns.append(column)

    if len(missing_columns) > 0:
        raise ValueError(
            "stg_jobicy_postings has no columns " + str(missing_columns)
            + ". Rebuild the database with sql/JobAvenir_01_Create_Database_MySQL.sql first."
        )

    # Check whether data was already imported
    cursor.execute(
        "SELECT COUNT(*) FROM stg_jobicy_postings"
    )

    existing_rows = cursor.fetchone()[0]

    if existing_rows > 0 and REPLACE_STAGING == 0:
        print(
            f"Import skipped: staging table already contains "
            f"{existing_rows} records."
        )

    else:
        if existing_rows > 0:
            # DELETE (not TRUNCATE) so a failed import rolls back to the old rows
            cursor.execute("DELETE FROM stg_jobicy_postings")
            print("Removed", existing_rows, "old staging rows")

        # SQL insert query
        insert_query = (
            "INSERT INTO stg_jobicy_postings ("
            + ", ".join(columns)
            + ") VALUES ("
            + ", ".join(["%s"] * len(columns))
            + ")"
        )

        # Convert DataFrame into tuples; missing values (e.g. no salary) become NULL.
        # Done row by row because df.where(..., None) leaves NaN in newer pandas.
        records = []
        for row in df[columns].itertuples(index=False, name=None):
            clean_row = []
            for value in row:
                if pd.isna(value):
                    clean_row.append(None)
                else:
                    clean_row.append(value)
            records.append(tuple(clean_row))

        # Insert records
        cursor.executemany(
            insert_query,
            records
        )

        connection.commit()

        print("Jobicy data imported successfully!")

    # Verify total records
    cursor.execute(
        "SELECT COUNT(*) FROM stg_jobicy_postings"
    )

    total = cursor.fetchone()[0]

    print("Total records in MySQL:", total)

except Exception as e:
    connection.rollback()
    print("Import failed:", e)
    raise

finally:
    cursor.close()

Step 4: Import the job-skills CSV into MySQL

In [ ]:
# Job-skill links extracted in 02_Jobicy_Data_Collection.ipynb
skills_path = CLEANED_FOLDER / "job_skills_clean.csv"

skills_df = pd.read_csv(skills_path, dtype=str)

print("Total Job-Skill Records:", len(skills_df))

skill_columns = ["job_id", "skill_name"]

assert list(skills_df.columns) == skill_columns, "job_skills_clean.csv columns do not match!"

cursor = connection.cursor()

try:
    cursor.execute("SELECT COUNT(*) FROM stg_job_skills")
    existing_rows = cursor.fetchone()[0]

    if existing_rows > 0 and REPLACE_STAGING == 0:
        print(f"Import skipped: stg_job_skills already contains {existing_rows} records.")

    else:
        if existing_rows > 0:
            cursor.execute("DELETE FROM stg_job_skills")
            print("Removed", existing_rows, "old staging rows")

        records = list(skills_df[skill_columns].itertuples(index=False, name=None))

        cursor.executemany(
            "INSERT INTO stg_job_skills (job_id, skill_name) VALUES (%s, %s)",
            records
        )

        connection.commit()

        print("Job-skill data imported successfully!")

    cursor.execute("SELECT COUNT(*) FROM stg_job_skills")
    print("Total records in MySQL:", cursor.fetchone()[0])

except Exception as e:
    connection.rollback()
    print("Import failed:", e)
    raise

finally:
    cursor.close()